In [ ]:
import os

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import loading as ld
import processing as prc
import visualization as viz

from matplotlib.transforms import ScaledTranslation
from matplotlib.lines import Line2D

plt.rcParams['legend.handlelength'] = 1
plt.rcParams['legend.handleheight'] = 1.125
plt.rcParams["font.family"] = "Avenir"

path_to_images = '/Users/Guille/Desktop/india_power/images'
path_to_output = '/Users/Guille/Desktop/india_power/output_csvs'
path_to_csvs = '/Users/Guille/Desktop/india_power/gridpath_india_viz/csvs'

In [ ]:
tech_labels_ = pd.read_csv(path_to_csvs + '/technology_labels.csv')

In [ ]:
# Grab data from databases for plotting energy dispatch and clean energy targets
def _load_losses(scen_labels_):

    # Load energy dispatch table and process data from database
    def __load_stor_losses_from_csv(df_, scenario):
        
        techs_ = ['Battery', 'Hydro_Pumped', 'Hydrogen']
        
        df_['power_mw'] = df_['number_of_hours_in_timepoint']*df_['timepoint_weight']*df_['power_mw']

        df_  = df_[['period', 'technology', 'load_zone', 'power_mw']]
        idx_ = df_['technology'].isin(techs_)
        
        df_.loc[idx_, 'technology']  = 'stor_losses'
        df_.loc[~idx_, 'technology'] = 'gen'
        
        df_ = df_.groupby(['period', 
                           'technology', 
                           'load_zone']).agg({'power_mw': 'sum'}).reset_index(drop = False)
        
        df_['scenario'] = scenario
                
        return df_

    def __load_tx_losses_from_csv(df_, scenario):
        
        df_['transmission_losses_lz_to']   = - df_['number_of_hours_in_timepoint']*df_['timepoint_weight']*df_['transmission_losses_lz_to'] 
        df_['transmission_losses_lz_from'] = - df_['number_of_hours_in_timepoint']*df_['timepoint_weight']*df_['transmission_losses_lz_from']

        df_1_ = df_[['period', 'load_zone_to', 'transmission_losses_lz_to']].copy()
        df_2_ = df_[['period', 'load_zone_from', 'transmission_losses_lz_from']].copy()

        df_1_['technology'] = 'tx_losses'
        df_2_['technology'] = 'tx_losses'

        df_1_ = df_1_.rename(columns = {'transmission_losses_lz_to': 'power_mw', 
                                        'load_zone_to': 'load_zone'})

        df_2_ = df_2_.rename(columns = {'transmission_losses_lz_from': 'power_mw',
                                        'load_zone_from': 'load_zone'})
        
        df_ = pd.concat([df_1_, df_2_], axis = 0)
        df_ = df_.groupby(['period', 
                           'technology', 
                           'load_zone']).agg({'power_mw': 'sum'}).reset_index(drop = False)

        df_['scenario'] = scenario
        
        return df_

    dfs_ = []
    # Open connection: open database and grab metadata
    for scen, path in zip(scen_labels_['scenario'], scen_labels_['path']):
        print(scen, path)
        
        dir_name  = r'{}/{}'.format(path, scen)
        dispatch_ = pd.read_csv(dir_name + f'/results/project_timepoint.csv', low_memory = False)
        tx_losses_ = pd.read_csv(dir_name + f'/results/transmission_timepoint.csv', low_memory = False)

        dfs_ += [__load_stor_losses_from_csv(dispatch_, scen)]
        dfs_ += [__load_tx_losses_from_csv(tx_losses_, scen)]

    return pd.concat(dfs_, axis = 0).reset_index(drop = True)


# Grab data from databases for plotting LCOE emissions
def _load_land_use(scen_labels_):
 
    # Load energy dispatch table and process data from database
    def __load_dispatch_from_csv(df_, scenario):
        
        df_['power_mw'] = df_['number_of_hours_in_timepoint']*df_['timepoint_weight']*df_['power_mw']

        df_ = df_[['period', 
                   'technology', 
                   'load_zone', 
                   'power_mw']]
            
        df_ = df_.groupby(['period', 
                           'technology', 
                           'load_zone']).agg({'power_mw': 'sum'}).reset_index(drop = False)
        
        df_['scenario'] = scenario
                
        return df_

    # Load fix capacity cost table and process data from database
    def __load_capacity_from_csv(df_, scenario):  

        df_ = df_[['period', 
                   'technology', 
                   'load_zone', 
                   'capacity_mw', 
                   'energy_capacity_mwh']]

        df_ = df_.groupby(['period', 
                           'technology', 
                           'load_zone']).agg({'capacity_mw': 'sum', 
                                              'energy_capacity_mwh': 'sum'})

        df_ = df_.reset_index(drop = False).rename(columns = {'energy_capacity_mwh': 'capacity_mwh'})

        df_['scenario'] = scenario

        return df_

    dfs_ = []
    # Open connection: open database and grab meta-data
    for scen, path in zip(scen_labels_['scenario'], scen_labels_['path']):
        print(scen, path)
        dir_name = r'{}/{}'.format(path, scen)        

        # Load fix costs from CSV files
        project_ = pd.read_csv(dir_name + f'/results/project_period.csv', low_memory = False)
        df_1_    = __load_capacity_from_csv(project_, scen)

        # Load variable costs from CSV files
        dispatch_ = pd.read_csv(dir_name + f'/results/project_timepoint.csv', low_memory = False)
        df_2_     = __load_dispatch_from_csv(dispatch_, scen)

        dfs_ += [pd.merge(df_1_, df_2_, on = ['scenario', 'technology', 'period', 'load_zone'])]
    
    df_                  = pd.concat(dfs_, axis = 0).reset_index(drop = True)
    df_['period']        = df_['period'].astype(int)
    df_['capacity_mw']   = df_['capacity_mw'].astype(float)
    df_['capacity_mwh']  = df_['capacity_mwh'].astype(float)
    df_['power_mw']      = df_['power_mw'].astype(float)

    return df_

In [ ]:
def _group_land_use_by_zone(df_, scen_labels_, land_use_, columns_ = ['period', 
                                                                      'scenario', 
                                                                      'technology', 
                                                                      'load_zone']):
    
    dfs_ = []
    # Open connection: open database and grab metadata
    for scen, zone in zip(scen_labels_['scenario'], scen_labels_['zone']):
        
        df_p_ = df_.loc[df_['scenario'] == scen].copy()
        
        idx_ = df_p_['load_zone'] == zone
        
        if idx_.sum() == 0.:
            df_p_['load_zone'] = 'all'
        else:
            df_p_ = df_p_.loc[idx_].reset_index(drop = True)
        
        dfs_.append(df_p_.groupby(columns_).sum().reset_index(drop = False))
        
    df_                 = pd.concat(dfs_, axis = 0).reset_index(drop = True)
    df_['period']       = df_['period'].astype(int)
    df_['capacity_mw']  = df_['capacity_mw'].astype(float)
    df_['capacity_mwh'] = df_['capacity_mwh'].astype(float)
    df_['power_mw']     = df_['power_mw'].astype(float)

    df_['area_m2'] = 0.
    for tech in df_['technology'].unique():
        idx_ = df_['technology'] == tech        
        df_.loc[idx_, 'area_m2'] = land_use_.loc[tech, 'land_use_intensity'] * df_.loc[idx_, land_use_.loc[tech, 'type']]


    land_use_['land_use_intensity'] = land_use_['land_use_intensity'].astype(float)
    
    return df_

In [ ]:
# Plot syestem losses (%) for different scenarios
def _plot_losses(ax, df_, scen_labels_, 
                 legend = False,
                 title  = '', legend_title = 'Scenario', y_min = None, y_max = None):
    
    for scen, zone in zip(scen_labels_['scenario'], scen_labels_['zone']):    

        data_ = df_.loc[(df_['scenario'] == scen) & (df_['load_zone'] == zone)].copy()   
        data_ = data_.sort_values(by = ['period'])

        x_ = data_.loc[data_['technology'] == 'gen', 'period'].to_numpy()
        y_ = data_.loc[data_['technology'] == 'tx_losses', 'power_mw'].to_numpy()
        z_ = data_.loc[data_['technology'] == 'stor_losses', 'power_mw'].to_numpy()
        w_ = data_.loc[data_['technology'] == 'gen', 'power_mw'].to_numpy()

        ax.plot(x_, - 100.*(y_ + z_)/w_, 
                color     = scen_labels_.loc[scen_labels_['scenario'] == scen, 'color'].to_numpy()[0], 
                linestyle = scen_labels_.loc[scen_labels_['scenario'] == scen, 'linestyle'].to_numpy()[0],
                label     = scen_labels_.loc[scen_labels_['scenario'] == scen, 'label'].to_numpy()[0],
                marker    = scen_labels_.loc[scen_labels_['scenario'] == scen, 'marker'].to_numpy()[0],
                zorder    = scen_labels_.loc[scen_labels_['scenario'] == scen, 'order'].to_numpy()[0],
                linewidth = scen_labels_.loc[scen_labels_['scenario'] == scen, 'linewidth'].to_numpy()[0], 
                alpha     = 0.75)
    
    ax.set_xticks(x_, x_)
    ax.xaxis.set_tick_params(labelsize = 14)
    ax.yaxis.set_tick_params(labelsize = 14)
    ax.set_ylabel(r'System Losses (%)', fontsize = 18)
    #ax.yaxis.set_major_formatter(mpl.ticker.StrMethodFormatter('{x:,.0f}'))

    if (y_min != None) & (y_max != None):
        ax.set_ylim(y_min, y_max)
        
    if legend:
        ax.legend(loc            = 'center left', 
                  title          = legend_title,
                  handlelength   = 1.75,
                  bbox_to_anchor = (1, 0.5), 
                  frameon        = False,
                  title_fontsize = 18,
                  prop           = {'size': 16})
    
    ax.set_title(title, fontsize = 18, 
                        y        = 0.9125)
    
# Plot land use (MHa) for different scenarios
def _plot_land_use(ax, df_, scen_labels_, 
                   ylabel       = r'Land Use (MHa)',
                   units        = 1e6,
                   legend       = False,
                   title        = '', 
                   legend_title = 'Scenario', 
                   y_min        = None, 
                   y_max        = None):
    
    df_p_ = df_[['period', 'scenario', 'load_zone', 'area_m2']].copy()
    df_p_ = df_p_.groupby(['period', 'scenario', 'load_zone']).sum().reset_index(drop = False)

    for scen, zone in zip(scen_labels_['scenario'], scen_labels_['zone']):    

        data_ = df_p_.loc[(df_p_['scenario'] == scen) & (df_p_['load_zone'] == zone)].copy()   
        data_ = data_.sort_values(by = ['period'])

        x_ = data_['period'].to_numpy()
        y_ = data_['area_m2'].to_numpy()

        ax.plot(x_, 0.0001*y_/units, 
                color     = scen_labels_.loc[scen_labels_['scenario'] == scen, 'color'].to_numpy()[0], 
                linestyle = scen_labels_.loc[scen_labels_['scenario'] == scen, 'linestyle'].to_numpy()[0],
                label     = scen_labels_.loc[scen_labels_['scenario'] == scen, 'label'].to_numpy()[0],
                marker    = scen_labels_.loc[scen_labels_['scenario'] == scen, 'marker'].to_numpy()[0],
                zorder    = scen_labels_.loc[scen_labels_['scenario'] == scen, 'order'].to_numpy()[0],
                linewidth = scen_labels_.loc[scen_labels_['scenario'] == scen, 'linewidth'].to_numpy()[0], 
                alpha     = 0.75)
    
    ax.set_xticks(x_, x_)
    ax.xaxis.set_tick_params(labelsize = 14)
    ax.yaxis.set_tick_params(labelsize = 14)
    ax.set_ylabel(ylabel, fontsize = 18)
    #ax.yaxis.set_major_formatter(mpl.ticker.StrMethodFormatter('{x:,.0f}'))

    if (y_min != None) & (y_max != None):
        ax.set_ylim(y_min, y_max)
        
    if legend:
        ax.legend(loc            = 'center left', 
                  title          = legend_title,
                  bbox_to_anchor = (1, 0.5),
                  handlelength   = 1.75,
                  frameon        = False,
                  title_fontsize = 18,
                  prop           = {'size': 16})
    
    ax.set_title(title, fontsize = 18, 
                        y        = 0.9125)
    
# Plot available land (%) for different scenarios
def _plot_available_land(ax, df_, scen_labels_, 
                         ylabel       = r'Total Land Used Area (%)',
                         units        = 1e6,
                         legend       = False,
                         title        = '',
                         legend_title = 'Scenario', 
                         y_min        = None, 
                         y_max        = None):
    
    df_p_ = df_[['period', 'scenario', 'load_zone', 'area_m2']].copy()
    df_p_ = df_p_.groupby(['period', 'scenario', 'load_zone']).sum().reset_index(drop = False)

    for scen, zone in zip(scen_labels_['scenario'], scen_labels_['zone']):    

        data_ = df_p_.loc[(df_p_['scenario'] == scen) & (df_p_['load_zone'] == zone)].copy()   
        data_ = data_.sort_values(by = ['period'])

        x_ = data_['period'].to_numpy()
        y_ = data_['area_m2'].to_numpy()

        ax.plot(x_, 100.*(0.0001*y_/units)/328.7, 
                color     = scen_labels_.loc[scen_labels_['scenario'] == scen, 'color'].to_numpy()[0], 
                linestyle = scen_labels_.loc[scen_labels_['scenario'] == scen, 'linestyle'].to_numpy()[0],
                label     = scen_labels_.loc[scen_labels_['scenario'] == scen, 'label'].to_numpy()[0],
                marker    = scen_labels_.loc[scen_labels_['scenario'] == scen, 'marker'].to_numpy()[0],
                zorder    = scen_labels_.loc[scen_labels_['scenario'] == scen, 'order'].to_numpy()[0],
                linewidth = scen_labels_.loc[scen_labels_['scenario'] == scen, 'linewidth'].to_numpy()[0], 
                alpha     = 0.75)
    
    ax.set_xticks(x_, x_)
    ax.xaxis.set_tick_params(labelsize = 14)
    ax.yaxis.set_tick_params(labelsize = 14)
    ax.set_ylabel(ylabel, fontsize = 18)
    #ax.yaxis.set_major_formatter(mpl.ticker.StrMethodFormatter('{x:,.0f}'))

    if (y_min != None) & (y_max != None):
        ax.set_ylim(y_min, y_max)
        
    if legend:
        ax.legend(loc            = 'center left', 
                  title          = legend_title,
                  handlelength   = 1.75,
                  bbox_to_anchor = (1, 0.5), 
                  frameon        = False,
                  title_fontsize = 18,
                  prop           = {'size': 16})
    
    ax.set_title(title, fontsize = 18, 
                        y        = 0.9125)

In [ ]:
path_to_data = '/Users/Guille/Desktop/india_power/gridpath_india_viz/data'

land_use_intensity_ = pd.read_csv(path_to_data + r'/land_use.csv', index_col = 0)
land_use_           = _load_land_use(scen_labels_)
grouped_land_use_   = _group_land_use_by_zone(land_use_, scen_labels_, land_use_intensity_)
losses_             = _load_losses(scen_labels_)
grouped_losses_     = _group_clean_energy_by_zone(losses_, scen_labels_)

In [ ]:
land_use_           = _load_land_use(scen_labels_)
land_use_intensity_ = pd.read_csv(path_to_data + r'/land_use.csv', index_col = 0)
grouped_land_use_   = _group_land_use_by_zone(land_use_, scen_labels_, land_use_intensity_)

In [ ]:
fig = plt.figure(figsize = (4., 5))
_ax  = plt.subplot(111)
    
# Plot land use for different scenarios
_plot_land_use(_ax, grouped_land_use_, scen_labels_, 
               ylabel = r'Land Used (MHa)',
               legend = True)

plt.savefig(path_to_images + '/land_use.png', 
            bbox_inches = 'tight', 
            dpi         = 300)

plt.show()

In [ ]:
fig = plt.figure(figsize = (3.5, 5))
_ax  = plt.subplot(111)
    
# Plot land use for different scenarios
_plot_available_land(_ax, grouped_land_use_, scen_labels_, 
                     ylabel = r'Total Land Used Area (%)',
                     legend = True)

plt.savefig(path_to_images + '/available_land.png', 
            bbox_inches = 'tight', 
            dpi         = 300)

plt.show()